# MERIT: Colab / Kaggle quick start

Enable **GPU**, then **Run all**. On Kaggle also enable **Internet**.
No Drive mount, credentials, dataset upload or CUDA installation is needed.
The notebook reuses the runtime's CUDA-enabled PyTorch and installs only missing
small project dependencies. A fresh cloud session may need those dependencies again.

This notebook tests **full-slice 512x512 feasibility**, with synthetic data,
batch size 1, AMP, and the complete MERIT architecture. It does not train on
patients or compare segmentation accuracy. A GPU with 16 GB is not guaranteed to fit.
Two GPUs do not automatically pool their memory; this notebook uses GPU 0.

The public source commit below is pinned for reproducibility. Its model code is
the same as the local native-512 experiment; unpublished local runtime/data fixes
are not fetched. This is a profiling notebook, not a full training/resume entry point.

In [ ]:
# Defaults are ready for Run all. No edits needed for the 512x512 experiment.
REPOSITORY = "https://github.com/torusama/merit-covid-ct.git"
SOURCE_COMMIT = "e3493dd9f2dac704d2def6225d76f19df3262ec4"
GRID_SIZE = 512
RUN_128_SANITY = True
CUDA_MEMORY_FRACTION = 0.9  # Leave room for CUDA libraries and the notebook service.

## 1. Check the assigned GPU
This short-lived child process releases its CUDA context on exit. The notebook
kernel itself does not import Torch or hold model/optimizer tensors.

In [ ]:
import json
import subprocess
import sys
from importlib import metadata
from packaging.specifiers import SpecifierSet

def runtime_probe():
    source = """import json, torch
available = torch.cuda.is_available()
print(json.dumps({
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'cuda_available': available,
    'gpus': [{'name': torch.cuda.get_device_name(i),
              'vram_mib': torch.cuda.get_device_properties(i).total_memory / 2**20}
             for i in range(torch.cuda.device_count())]}))
"""
    result = subprocess.run([sys.executable, "-c", source], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("The runtime's PyTorch could not load. Select a current GPU runtime.\n" + result.stderr)
    info = json.loads(result.stdout)
    if not info["cuda_available"]:
        raise RuntimeError("No CUDA GPU assigned. Enable a GPU in the notebook settings, reconnect, and Run all.")
    if not SpecifierSet(">=2.4.1").contains(info["torch"], prereleases=True):
        raise RuntimeError("This project needs Torch >=2.4.1. Use a newer runtime; this notebook will not replace Torch/CUDA.")
    return info

ENVIRONMENT = runtime_probe()
print(json.dumps(ENVIRONMENT, indent=2))

## 2. Fetch code and reuse installed packages
Code is cached inside the current runtime. No dataset is downloaded or inspected.
Torch and NumPy versions are constrained to the versions already supplied by the
platform; the setup cannot silently replace them. All model runs use fresh child
processes, so installing a missing library does not require restarting the kernel.

In [ ]:
from datetime import datetime, timezone
import os
from pathlib import Path
import uuid

BASE = (Path(os.environ["MERIT_CLOUD_HOME"]) if os.environ.get("MERIT_CLOUD_HOME") else
        Path("/kaggle/working") if Path("/kaggle/working").is_dir() else
        Path("/content") if Path("/content").is_dir() else Path.cwd() / "outputs" / "cloud_quickstart")
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / ("merit-source-" + SOURCE_COMMIT[:12])
RUN = BASE / "merit-results" / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8])
RUN.mkdir(parents=True, exist_ok=False)

def git(*args, check=True):
    return subprocess.run(["git", "-C", str(PROJECT), *args], check=check,
                          capture_output=True, text=True)

if not PROJECT.exists():
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPOSITORY, str(PROJECT)], check=True)
elif not (PROJECT / ".git").is_dir():
    raise RuntimeError(f"Source path exists but is not a Git checkout: {PROJECT}")
if git("status", "--porcelain", "--untracked-files=no").stdout.strip():
    raise RuntimeError("The cached source has local edits. Save those edits and select a fresh source directory.")
if git("cat-file", "-e", SOURCE_COMMIT + "^{commit}", check=False).returncode:
    git("fetch", "--quiet", "--depth", "1", "origin", SOURCE_COMMIT)
git("checkout", "--quiet", "--detach", SOURCE_COMMIT)
assert git("rev-parse", "HEAD").stdout.strip() == SOURCE_COMMIT
assert (PROJECT / "scripts/merit_preflight.py").is_file()

requirements = {
    "monai": "==1.5.2", "einops": "==0.8.2", "numpy": ">=1.24,<3",
    "nibabel": ">=5", "scipy": ">=1.10", "PyYAML": ">=6",
}
missing = []
for name, bounds in requirements.items():
    try:
        installed = metadata.version(name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or not SpecifierSet(bounds).contains(installed, prereleases=True):
        missing.append(name + bounds)

protected = {name: metadata.version(name) for name in ("torch", "numpy")}
constraints = RUN / "runtime_constraints.txt"
constraints.write_text("".join(name + "==" + version + "\n" for name, version in protected.items()))
if missing:
    print("Installing only:", ", ".join(missing), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                    "--constraint", str(constraints), *missing], check=True)
else:
    print("All project dependencies already satisfy the requirements; skipping pip.")
for name, version in protected.items():
    if metadata.version(name) != version:
        raise RuntimeError(f"Protected runtime package changed unexpectedly: {name}")
ENVIRONMENT["packages"] = {name: metadata.version(name) for name in requirements}
ENVIRONMENT["source_commit"] = SOURCE_COMMIT
ENVIRONMENT["source_repository"] = REPOSITORY
(RUN / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")
print("Source:", PROJECT)
print("Reports:", RUN)

## 3. Profile the full architecture
Each size runs in its own process and releases GPU memory on exit. The 128x128
sanity check detects environment failures before the 512x512 attempt. Both use
feature size 16, depths 3/3/3/3, all TEP/MER/PCG modules, and the saved loss policy.
The effective 512 grid is set explicitly by the existing profiler, without
changing the experiment YAML on disk. No real images or labels are opened.

**PASS** means the requested complete synthetic optimizer steps succeeded.
**OOM** means insufficient memory in this setup; **FAIL** means another error.
Neither is an accuracy result. The final report is still available after OOM.

In [ ]:
if type(GRID_SIZE) is not int or GRID_SIZE < 32 or GRID_SIZE % 32:
    raise ValueError("GRID_SIZE must be a positive multiple of 32.")
if not 0 < CUDA_MEMORY_FRACTION <= 1:
    raise ValueError("CUDA_MEMORY_FRACTION must be in (0,1].")

worker = RUN / "profile_worker.py"
worker.write_text("""import json, os, sys
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
torch.cuda.set_per_process_memory_fraction(float(sys.argv[4]), torch.cuda.current_device())
from merit_covid.engine.preflight import profile_model
report = profile_model(sys.argv[1], sys.argv[2], size=int(sys.argv[3]), batch_size=1,
                       device='cuda', warmup=1, repeats=1)
report['cloud_allocator_fraction'] = float(sys.argv[4])
Path(sys.argv[2]).write_text(json.dumps(report, indent=2, allow_nan=False))
raise SystemExit(0 if report['status'] == 'PASS' else 1)
""")
environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT / "src") + os.pathsep + environment.get("PYTHONPATH", "")
results = []
sizes = ([128] if RUN_128_SANITY and GRID_SIZE != 128 else []) + [GRID_SIZE]
for size in sizes:
    destination = RUN / f"profile_{size}.json"
    command = [sys.executable, str(worker), str(PROJECT / "configs/merit_index.yaml"),
               str(destination), str(size), str(CUDA_MEMORY_FRACTION)]
    print(f"Starting {size}x{size}, batch 1, AMP, complete synthetic training steps...", flush=True)
    with (RUN / f"profile_{size}.log").open("w") as log:
        process = subprocess.Popen(command, cwd=PROJECT, env=environment, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        exit_code = process.wait()
    if destination.exists():
        report = json.loads(destination.read_text())
    else:
        report = {"status": "FAIL", "error": "Worker produced no report; inspect its log.", "optimizer_steps": 0}
    row = {"grid": size, "status": report["status"], "exit_code": exit_code,
           "optimizer_steps": report.get("optimizer_steps", 0),
           "peak_allocated_mib": report.get("peak_allocated_mib"),
           "peak_reserved_mib": report.get("peak_reserved_mib"),
           "mean_step_seconds": report.get("mean_step_seconds"),
           "parameter_count": report.get("parameter_count"), "error": report.get("error")}
    results.append(row)
    (RUN / "summary.json").write_text(json.dumps(results, indent=2) + "\n")
    print(json.dumps(row, indent=2), flush=True)
    if size != GRID_SIZE and report["status"] != "PASS":
        print("Sanity check failed; the 512-grid attempt was not started. Read the saved report.")
        break

## 4. Save the report
The ZIP contains GPU/software information, source commit, effective configs,
logs, successful optimizer-step counts, timing and memory/error reports. Kaggle
stores it under `/kaggle/working`; use **Save Version** to retain outputs. On
Colab, run the download cell below before the runtime is discarded. If the
browser does not save the file, open the left Files panel, refresh, expand
`merit-results`, then use the ZIP file's three-dot menu > Download. The printed
`/content/...` path is a runtime filesystem path, not a browser download URL.
No patient data, trained model or credentials are included.

If 512 passes, the next step is a bounded real-CT pilot before full training.
Do not interpret synthetic throughput or loss as segmentation quality.

In [ ]:
import shutil
archive = Path(shutil.make_archive(str(RUN), "zip", root_dir=RUN))
print("Report ZIP:", archive)
print("Download using the next cell, or the Files panel > ZIP menu > Download.")

In [ ]:
# On Colab this downloads the completed report. On Kaggle use the Output files.
try:
    import google.colab
except ImportError:
    print("Open the report ZIP from the notebook's output files:", archive)
else:
    from google.colab import files
    files.download(str(archive))
    print("If no download appears, open Files on the left, refresh, then merit-results > ZIP menu > Download.")